# Phoenix Perps — Funding Rate Analysis

Exploratory analysis of SOL perpetual funding rates across venues.
Data sources: Binance FAPI, Drift Protocol.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from utils import annualize_8h_rate

%matplotlib inline
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# Load data
binance = pd.read_csv('../data/binance_sol_funding.csv', parse_dates=['timestamp'])
print(f'Binance: {len(binance)} records from {binance.timestamp.iloc[0]} to {binance.timestamp.iloc[-1]}')
binance.head()

In [ ]:
# Compute annualized APY
binance['apy'] = binance['rate_8h'].apply(annualize_8h_rate)
binance['apy_pct'] = binance['apy'] * 100

# Plot funding rates over time
fig, ax = plt.subplots()
ax.plot(binance['timestamp'], binance['apy_pct'], linewidth=0.5, alpha=0.8, color='#3b82f6')
ax.axhline(y=15, color='red', linestyle='--', alpha=0.7, label='Entry threshold (15%)')
ax.axhline(y=2, color='orange', linestyle='--', alpha=0.7, label='Exit threshold (2%)')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.set_ylabel('Annualized APY (%)')
ax.set_title('Binance SOL-PERP Funding Rate (Annualized)')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of funding rates
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(binance['rate_8h'] * 100, bins=50, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('8h Funding Rate (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of 8h Rates')
axes[0].axvline(x=0, color='red', linestyle='--')

axes[1].hist(binance['apy_pct'], bins=50, color='#22c55e', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Annualized APY (%)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Annualized APY')
axes[1].axvline(x=15, color='red', linestyle='--', label='Entry')
axes[1].axvline(x=2, color='orange', linestyle='--', label='Exit')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Key statistics
print('=== Funding Rate Statistics ===')
print(f'Mean 8h rate:        {binance.rate_8h.mean()*100:.4f}%')
print(f'Median 8h rate:      {binance.rate_8h.median()*100:.4f}%')
print(f'Std dev 8h rate:     {binance.rate_8h.std()*100:.4f}%')
print(f'Mean annualized APY: {binance.apy.mean()*100:.2f}%')
print(f'% positive rates:    {(binance.rate_8h > 0).mean()*100:.1f}%')
print(f'% above 15% APY:     {(binance.apy > 0.15).mean()*100:.1f}%')
print(f'Max 8h rate:         {binance.rate_8h.max()*100:.4f}%')
print(f'Min 8h rate:         {binance.rate_8h.min()*100:.4f}%')

In [ ]:
# Rolling average funding rate
binance['rolling_apy_7d'] = binance['apy_pct'].rolling(21).mean()  # 21 periods = 7 days
binance['rolling_apy_30d'] = binance['apy_pct'].rolling(90).mean()  # 90 periods = 30 days

fig, ax = plt.subplots()
ax.plot(binance['timestamp'], binance['rolling_apy_7d'], label='7-day MA', color='#3b82f6', linewidth=1.5)
ax.plot(binance['timestamp'], binance['rolling_apy_30d'], label='30-day MA', color='#f97316', linewidth=1.5)
ax.axhline(y=15, color='red', linestyle='--', alpha=0.5, label='Entry')
ax.axhline(y=2, color='orange', linestyle='--', alpha=0.5, label='Exit')
ax.axhline(y=0, color='gray', alpha=0.3)
ax.set_ylabel('Annualized APY (%)')
ax.set_title('Rolling Average Funding Rate')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()